<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_0_0_dataset_preparation_mnq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_0_0_dataset_preparation_mnq

## Resumen

Esta notebook realiza la **preparación inicial del dataset intradía del MNQ (Micro E-mini Nasdaq 100)**.  
El objetivo es construir un dataset limpio, consistente y estructurado que servirá como base para la ingeniería de factores y el entrenamiento de modelos.

0. **Configuración del entorno**
   - Clonado del repositorio y montaje de Google Drive.
   - Instalación e importación de librerías necesarias.

1. **Fuente de datos**
   - Datos históricos intradía del MNQ (OHLCV, frecuencia de 1 minuto) exportados desde NinjaTrader.
   - Archivos originales en formato `.txt`, en zona horaria UTC.

2. **Generación del dataset**
   - Unificación de todos los archivos `.txt` en un único DataFrame.
   - Asignación de nombres de columnas: `open`, `high`, `low`, `close`, `volume`.
   - Conversión de la columna `datetime` a índice temporal.

3. **Filtrado**
   - Conserva solo **días hábiles bursátiles** (se eliminan fines de semana y feriados de mercado de EE.UU.).
   - Conversión de marcas de tiempo de **UTC → US/Eastern**.
   - Filtrado de **horario de mercado** (09:30–16:00) más pre-market (desde 08:30).

4. **Validación de registros diarios**
   - Verificación de que cada día contenga la cantidad esperada de registros minuto a minuto.
   - Detección y eliminación de días incompletos o con irregularidades.

5. **Chequeo de continuidad temporal**
   - Confirmación de que los datos intradía estén en intervalos consecutivos de 1 minuto, sin gaps.

6. **Dataset final**
   - Guardado del dataset limpio en formato `.parquet` dentro de Google Drive.

---

**Resultado:** Un dataset intradía del MNQ completamente limpio y estandarizado, listo para la ingeniería de factores y el modelado.

## 0. Configuración del Entorno

### 0.1. Clonado de repositorio / Acceso a Drive

In [25]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


In [26]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías

In [27]:
import sys
!{sys.executable} -m pip install -q pandas_market_calendars
print("✅ Librería instalada: pandas_market_calendars")

✅ Librería instalada: pandas_market_calendars


### 0.3. Importación de librerías

In [28]:
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
from tabulate import tabulate

# Calendario de mercados
import pandas_market_calendars as mcal
import pandas as pd
import requests
from io import StringIO


### 0.4. Acceso a archivos locales/remotos

## 1. Contexto y fuente de datos

Los datos corresponden al contrato MNQ (Micro E-mini Nasdaq 100) descargados desde NinjaTrader con frecuencia de un minuto (formato OHLCV).

- Open: precio de apertura
- High: precio máximo
- Low: precio mínimo
- Close: precio de cierre
- Volume: volumen negociado

Los datos están en la zona horaria UTC.


## 2. Generación de dataset desde archivos históricos

Dado que los contratos se encuentran almacenados en archivos .txt dentro de la carpeta historicos_mnq, es necesario unificarlos en un único dataset consolidado.

La siguiente función se encarga de leer los archivos .txt, asignar nombres a las columnas correspondientes y establecer la columna datetime como índice temporal del dataframe.

In [48]:
def generar_df ():

    # Ruta a los archivos .txt
    ruta_historicos_repositorio = '/content/neural_profit/0_mnq_historical_data/*.txt'  # Reemplace con su ruta local
    ruta_historicos_drive = f'{drive_path}/0_mnq_historical_data/*.txt'

    # Determinar qué ruta usar
    if glob.glob(ruta_historicos_drive):
        print("📂 Usando históricos desde Google Drive")
        ruta_historicos = ruta_historicos_drive

    elif glob.glob(ruta_historicos_repositorio):
        print("📂 Usando históricos desde el repositorio local")
        ruta_historicos = ruta_historicos_repositorio
    else:
        raise FileNotFoundError("No se encontraron archivos históricos en el repositorio ni en el Drive.")

    # Lista para almacenar DataFrames individuales
    df_mnq = []

    # Leer todos los archivos .txt
    for archivo in glob.glob(ruta_historicos):
        df = pd.read_csv(
            archivo,
            sep=';',
            header=None,
            names=['datetime', 'open', 'high', 'low', 'close', 'volume'],
            dtype={'open': float, 'high': float, 'low': float, 'close': float, 'volume': int}
        )

        # Convertir columna 'datetime' al formato datetime real
        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y%m%d %H%M%S')

        # Establecer como índice
        df.set_index('datetime', inplace=True)

        df_mnq.append(df)

    # Unir todos los DataFrames
    df_mnq_raw = pd.concat(df_mnq)
    # Ordenar por fecha si es necesario
    df_mnq_raw.sort_index(inplace=True)

    return df_mnq_raw

El siguiente bloque de código verifica si el dataset consolidado ya ha sido generado previamente.

En particular, comprueba la existencia del archivo mnq_raw_data.parquet.

- Si el archivo está presente, se carga directamente en la variable df_mnq.

- En caso contrario, se invoca la función generate_dataset() para generar el dataset a partir de los archivos originales.

In [49]:
def load_or_build_raw_dataset():

    mnq_raw_data_file= f'{drive_path}/1_mnq_dataset_preparation/mnq_raw_data.parquet'

    if os.path.exists(mnq_raw_data_file):
        print("📂 Archivo encontrado en disco. Cargando dataset local...")
        df_mnq_raw = pd.read_parquet(mnq_raw_data_file)

    else:
        print("⚠️ Archivo no encontrado. Generando dataset desde archivos históricos...")
        df_mnq_raw = generar_df()
        df_mnq_raw.to_parquet(mnq_raw_data_file, index=True)
        print("✅ Dataset generado y guardado localmente.")

    return df_mnq_raw

In [50]:
df_mnq_raw = load_or_build_raw_dataset()

📂 Archivo encontrado en disco. Cargando dataset local...


## 3. Filtrado de días no hábiles y horario bursátil

### 3.1. Filtrado de fines de semana y feriados bursátiles estadounidenses

Es necesario filtrar del conjunto de datos aquellas filas correspondientes a sábados, domingos y feriados bursátiles. Para ello, se utilizará la librería pandas_market_calendars, que permite identificar los días hábiles de operación según el calendario oficial del NASDAQ.

La función implementada filtra un DataFrame con índice de tipo DatetimeIndex, conservando únicamente aquellas filas cuya fecha coincida con un día hábil del mercado. La marca temporal completa (fecha y hora) se mantiene sin modificaciones.

In [51]:
def filtrar_dias_habiles_nasdaq(df):
    # Crear el calendario del mercado NASDAQ
    nasdaq = mcal.get_calendar('NASDAQ')

    # Obtener el rango de fechas del índice del DataFrame
    start_date = df.index.min().date()
    end_date = df.index.max().date()

    # Obtener el cronograma de días hábiles del mercado
    valid_dates = nasdaq.schedule(start_date=start_date, end_date=end_date).index.date

    # Filtrar el DataFrame verificando si la fecha de cada marca temporal está en los días válidos
    df_filtrado = df[df.index.normalize().isin(valid_dates)]

    return df_filtrado

In [52]:
df_mnq = filtrar_dias_habiles_nasdaq (df_mnq_raw)

### 3.2. Filtrado de horario de operación de mercado de New York (09:30 a 16:00) con pre mercado, desde las 08:00

Dado que los timestamps del índice (DatetimeIndex) provienen de archivos .txt sin información de zona horaria, es necesario indicar explícitamente a pandas que dichos valores están en formato UTC.

Una vez establecido el timezone, se procede a convertir los timestamps desde UTC a la hora local del mercado estadounidense (zona US/Eastern), correspondiente a los horarios de operación del NASDAQ/NYSE. Esta conversión se realiza teniendo en cuenta automáticamente los ajustes por horario de verano o invierno.

In [34]:
def configurar_zona_horaria(df, from_tz='UTC', to_tz='America/New_York'):
    """
    Asegura que el índice del DataFrame tenga zona horaria 'from_tz'
    y lo convierte a la zona horaria 'to_tz'.
    """
    if df.index.tz is None:
        # Localiza en from_tz si no tiene zona horaria
        df.index = df.index.tz_localize(from_tz)
    # Convierte a la zona horaria deseada
    df.index = df.index.tz_convert(to_tz)
    return df

In [53]:
df_mnq = configurar_zona_horaria(df_mnq)

La siguiente función selecciona únicamente las muestras que se encuentran dentro del horario regular de operación bursátil del NASDAQ.

Filtra un DataFrame cuyo índice es de tipo DatetimeIndex, conservando solo aquellas filas cuya marca temporal se encuentre entre las 09:30 y 16:00 horas (US/Eastern), correspondientes al horario de negociación estándar en días hábiles de mercado.

Particularmente, decido agregar una hora de pre mercado, desde las 08:00AM.

In [54]:
def filtrar_horas_habiles_nasdaq(df, inicio: str, final:str): #'08:00:00', '16:00:00'
    # Filtrar solo las filas dentro de las horas de mercado (de 8:00 AM a 4:00 PM)
    #df_filtered = df.between_time('08:00:00', '16:00:00')
    df_filtered = df.between_time(inicio, final)

    # Retornar el DataFrame filtrado
    return df_filtered

In [55]:
df_mnq = filtrar_horas_habiles_nasdaq(df_mnq, '04:30:00', '16:00:00' )

In [56]:
df_mnq

,open,high,low,close,volume
datetime,,,,,
2019-12-23 04:30:00-05:00,8718.25,8718.75,8718.25,8718.50,13
2019-12-23 04:31:00-05:00,8718.50,8719.25,8718.50,8719.25,22
2019-12-23 04:32:00-05:00,8719.25,8720.00,8719.00,8719.75,22
2019-12-23 04:33:00-05:00,8719.50,8720.25,8719.50,8719.75,60
2019-12-23 04:34:00-05:00,8719.50,8719.50,8719.00,8719.25,43
...,...,...,...,...,...
2025-06-13 15:56:00-04:00,21624.50,21635.00,21613.50,21617.50,3251
2025-06-13 15:57:00-04:00,21616.50,21635.25,21615.75,21623.75,2201
2025-06-13 15:58:00-04:00,21623.25,21632.75,21616.50,21621.75,1859


## 4. Análisis de registros diarios

Es necesario verificar que todos los días del conjunto de datos contengan la misma cantidad de registros y que estos sean consecutivos, es decir, que no falte ningún minuto dentro de cada jornada.

La función analizar_registros_por_dia permite realizar este control sobre un DataFrame con índice de tipo datetime. La función contabiliza la cantidad de registros por día e imprime una tabla resumen que indica cuántos días presentan una determinada cantidad de registros. Esto resulta útil para identificar inconsistencias, como días incompletos o interrupciones en la frecuencia temporal esperada.

In [57]:
def analizar_registros_por_dia(df: pd.DataFrame) -> pd.Series:
    """
    Analiza la cantidad de registros por día en un DataFrame con índice datetime.

    Imprime:
    - Distribución de la cantidad de registros por día.
    - Porcentaje de días con menos registros que el valor más frecuente.

    Retorna:
    - Serie con el conteo de registros por cada día.
    """
    # Contar la cantidad de registros por día
    conteo_diario = df.groupby(df.index.date).size()

    # Calcular la distribución de registros por día
    distribucion = conteo_diario.value_counts().sort_index(ascending=False)
    tabla = [[registros, cantidad_dias] for registros, cantidad_dias in distribucion.items()]

    #print("Distribución de cantidad de registros por día:\n")
    #print(tabulate(tabla, headers=["Registros por día", "Cantidad de días"], tablefmt="grid"))

    # Determinar el valor más frecuente de registros por día
    registros_dia_completo = conteo_diario.mode().iloc[0]
    print(f"\nCantidad de registros en un día completo: {registros_dia_completo}")

    # Calcular el porcentaje de días incompletos
    total_dias = len(conteo_diario)
    dias_con_menos = (conteo_diario < registros_dia_completo).sum()
    porcentaje = (dias_con_menos / total_dias) * 100

    print(f"Días con menos de {registros_dia_completo} registros: {dias_con_menos} de {total_dias} ({porcentaje:.2f}%)")

    return conteo_diario, registros_dia_completo

In [58]:
resumen, registros_dia_completo = analizar_registros_por_dia(df_mnq)


Cantidad de registros en un día completo: 691
Días con menos de 691 registros: 73 de 1368 (5.34%)


Como podemos observar en la tabla, la gran mayoría de días tienen `481` muestras. Y representan más del 95% del total de los datos.


### 4.1. Filtrado de días incompletos

La siguiente función encuentra los indices de las fechas con registros incompletos:

In [59]:
def busqueda_fechas_incompletas(df: pd.DataFrame, registros_esperados: int, gap_minutes: int = 1,) -> list:

    """
    Muestra una tabla con los días que tienen menos de los registros esperados o presentan irregularidades temporales.
    Retorna una lista con esas fechas.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: intervalo esperado entre registros consecutivos (en minutos).
    - registros_esperados: cantidad esperada de registros por día.

    Retorna:
    - Lista de fechas (datetime.date) con menos registros de los esperados o problemas temporales.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()
    base_time_diff = pd.Timedelta(minutes=gap_minutes)

    conteos = df.groupby(df.index.date).size()
    fechas_problema = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        tiene_irregularidades = (time_diff != base_time_diff).any()
        cantidad = conteos[date]

        if cantidad < registros_esperados or tiene_irregularidades:
            fechas_problema.append((date, cantidad))

    '''
    if fechas_problema:
        print(f"\nDía con menos de {registros_esperados} registros o con problemas temporales:\n")
        print(f"{'+' + '-'*21 + '+' + '-'*20 + '+'}")
        print(f"| {'Fecha'.ljust(19)} | {'Registros'.rjust(18)} |")
        print(f"{'+' + '='*21 + '+' + '='*20 + '+'}")
        for fecha, registros in fechas_problema:
            print(f"| {str(fecha).ljust(19)} | {str(registros).rjust(18)} |")
            print(f"{'+' + '-'*21 + '+' + '-'*20 + '+'}")
    else:
        print("No se encontraron días con irregularidades ni registros incompletos.")
    '''
    # Solo devolver las fechas
    return [fecha for fecha, _ in fechas_problema]

Elimino las fechas con registros incompletos:

In [60]:
def eliminar_fechas_incompletas(df: pd.DataFrame, registros_esperados ):
  # Filtrar eliminando las fechas con problemas
  df = df[~df.index.to_series().dt.date.isin(busqueda_fechas_incompletas(df, registros_esperados))]
  analizar_registros_por_dia(df)
  return df

In [61]:
df_mnq = eliminar_fechas_incompletas(df_mnq, registros_dia_completo)


Cantidad de registros en un día completo: 691
Días con menos de 691 registros: 0 de 1295 (0.00%)


## 5. Verificación de continuidad temporal minuto a minuto

Es necesario verificar que los registros correspondientes a un mismo día estén dispuestos de forma consecutiva, con una separación exacta de un minuto entre cada muestra.

In [62]:
def detectar_gaps(df: pd.DataFrame, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"Se encontraron problemas en {len(problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for i in range(len(problem_indices)):
            idx = problem_indices[i][0]
            diff = df.loc[idx, 'time_diff']
            date = idx.date()
            count = conteos[date]
            print(f'\t{idx} -> Diferencia: {diff} | # Registros: {count}')
    else:
        print("No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.")

    return problem_indices

In [63]:
detectar_gaps(df_mnq)

No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.


[]

## 6. Guardado de dataset final



Se guarda un dataset compuesto por 1198 días, cada uno con 511 registros correspondientes a minutos consecutivos.

El conjunto de datos incluye únicamente días hábiles de operación bursátil, y abarca el intervalo horario comprendido entre las 07:30 y las 16:00 horas (US/Eastern).

In [64]:
df_mnq

,open,high,low,close,volume
datetime,,,,,
2020-01-02 04:30:00-05:00,8813.25,8813.25,8812.50,8813.25,13
2020-01-02 04:31:00-05:00,8812.50,8812.50,8811.00,8812.50,121
2020-01-02 04:32:00-05:00,8812.50,8813.25,8811.75,8811.75,53
2020-01-02 04:33:00-05:00,8812.00,8812.25,8810.50,8810.50,37
2020-01-02 04:34:00-05:00,8810.75,8812.25,8810.50,8812.00,36
...,...,...,...,...,...
2025-06-13 15:56:00-04:00,21624.50,21635.00,21613.50,21617.50,3251
2025-06-13 15:57:00-04:00,21616.50,21635.25,21615.75,21623.75,2201
2025-06-13 15:58:00-04:00,21623.25,21632.75,21616.50,21621.75,1859


In [65]:
# Ruta del dataset en Drive
mnq_intraday_data_file = f"{drive_path}/5_transformer_90_model/mnq_intraday_data.parquet"

# Verificar si el archivo ya existe
if os.path.exists(mnq_intraday_data_file):
    print(f"Archivo encontrado en Drive: {mnq_intraday_data_file}")
    df_mnq = pd.read_parquet(mnq_intraday_data_file)
    print("Dataset cargado desde Drive.")
else:
    print("No se encontró el archivo en Drive. Guardando nuevo dataset...")
    os.makedirs(os.path.dirname(mnq_intraday_data_file), exist_ok=True)
    df_mnq.to_parquet(mnq_intraday_data_file, index=True)
    print(f"Dataset guardado en: {mnq_intraday_data_file}")

No se encontró el archivo en Drive. Guardando nuevo dataset...
Dataset guardado en: /content/drive/MyDrive/neural_profit/5_transformer_90_model/mnq_intraday_data.parquet
